# Feature Store Experiement: using minmaxscaler 

### Source Material: 
[Build an End-to-End ML Workflow in Snowflake
](https://www.snowflake.com/en/developers/guides/end-to-end-ml-workflow/?index=..%2F..index#1)
and
https://docs.google.com/document/d/16XzBjQc5BXiacMPMICeB-dMbu5Y8hBMzEDLm_XcFNIQ/edit?tab=t.0

also
https://github.com/Snowflake-Labs/sfguide-getting-started-with-snowflake-feature-store/blob/main/best_practice_guide/featurestore_bestpractice_guide.pdf


In [ ]:
!pip install snowflake-ml-python scikit-learn polars streamlit

import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

# Snowpark session
from snowflake.snowpark import DataFrame
from snowflake.snowpark.functions import col, to_timestamp, min, max, month, dayofweek, dayofyear, avg, date_add, sql_expr
from snowflake.snowpark.types import IntegerType
from snowflake.snowpark import Window


import pandas as pd
import streamlit as st
from sklearn.preprocessing import MinMaxScaler

In [ ]:
USE DATABASE AICOLLEGE;
USE SCHEMA CUSTOMER_FEATURE_STORE;
USE ROLE AICOLLEGE;
USE WAREHOUSE AICOLLEGE;
SELECT CURRENT_ROLE();
SELECT CURRENT_WAREHOUSE();

In [ ]:
--select * from E2E_SNOW_MLOPS_DB.MLOPS_SCHEMA.MORTGAGE_LENDING_DEMO_DATA LIMIT 10;
CREATE OR REPLACE TABLE AICOLLEGE.CUSTOMER_FEATURE_STORE.MORTGAGE_LENDING_DEMO_DATA AS
SELECT * FROM E2E_SNOW_MLOPS_DB.MLOPS_SCHEMA.MORTGAGE_LENDING_DEMO_DATA;

In [ ]:
# https://docs.google.com/document/d/16XzBjQc5BXiacMPMICeB-dMbu5Y8hBMzEDLm_XcFNIQ/edit?tab=t.0

# === imports ===
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore
from snowflake.ml.data.data_connector import DataConnector

import polars as pl
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from snowflake.ml.registry import registry as model_registry
from snowflake.ml.model import model_signature



In [ ]:
#Update this VERSION_NUM to version your features, models etc!
VERSION_NUM = '0'
DB = "AICOLLEGE" 
SCHEMA = "CUSTOMER_FEATURE_STORE" 
COMPUTE_WAREHOUSE = "AICOLLEGE" 

## Preparing the data

In [ ]:
print("Reading table data...")
df = session.table("AICOLLEGE.PUBLIC.MORTGAGE_LENDING_DEMO_DATA")
df.show(5)

In [ ]:
df.select(min('TS'), max('TS'))

## Feature Engineering

In [ ]:
#Create a dict with keys for feature names and values containing transform code

feature_eng_dict = dict()

#Timstamp features
feature_eng_dict["TIMESTAMP"] = date_add(to_timestamp("TS"), -1)
feature_eng_dict["MONTH"] = month("TIMESTAMP")
feature_eng_dict["DAY_OF_YEAR"] = dayofyear("TIMESTAMP") 
feature_eng_dict["DOTW"] = dayofweek("TIMESTAMP")

# df= df.with_columns(feature_eng_dict.keys(), feature_eng_dict.values())

#Income and loan features
feature_eng_dict["LOAN_AMOUNT"] = col("LOAN_AMOUNT_000s")*1000
feature_eng_dict["INCOME"] = col("APPLICANT_INCOME_000s")*1000
feature_eng_dict["INCOME_LOAN_RATIO"] = col("INCOME")/col("LOAN_AMOUNT")

county_window_spec = Window.partition_by("COUNTY_NAME")
feature_eng_dict["MEAN_COUNTY_INCOME"] = avg("INCOME").over(county_window_spec)
feature_eng_dict["HIGH_INCOME_FLAG"] = (col("INCOME")>col("MEAN_COUNTY_INCOME")).astype(IntegerType())

feature_eng_dict["AVG_THIRTY_DAY_LOAN_AMOUNT"] =  sql_expr("""AVG(LOAN_AMOUNT) OVER (PARTITION BY COUNTY_NAME ORDER BY TIMESTAMP  
                                                            RANGE BETWEEN INTERVAL '30 DAYS' PRECEDING AND CURRENT ROW)""")

df = df.with_columns(feature_eng_dict.keys(), feature_eng_dict.values())
df.show(3)

## Feature Store

In [ ]:
# -----------------------------------------------
# 1. INITIALIZE SESSION & FEATURE STORE
# -----------------------------------------------

session = get_active_session()
fs = FeatureStore(
    session=session, 
    database=DB, 
    name=SCHEMA, 
    default_warehouse=COMPUTE_WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

In [ ]:
#define new entity
loan_id_entity = Entity(
    name = "LOAN_ENTITY",
    join_keys = ["LOAN_ID"],
    desc = "Features defined on a per loan level")
    #register
fs.register_entity(loan_id_entity)
print("Registered new entity")

In [ ]:
#Create a dataframe with just the ID, timestamp, and engineered features. We will use this to define our feature view
feature_df = df.select(["LOAN_ID"]+list(feature_eng_dict.keys()))
feature_df.show(5)

In [ ]:
#define and register feature view
loan_fv = FeatureView(
    name="Mortgage_Feature_View",
    entities=[loan_id_entity],
    feature_df=feature_df,
    timestamp_col="TIMESTAMP",
    refresh_freq="1 day")

#add feature level descriptions

loan_fv = loan_fv.attach_feature_desc(
    {
        "MONTH": "Month of loan",
        "DAY_OF_YEAR": "Day of calendar year of loan",
        "DOTW": "Day of the week of loan",
        "LOAN_AMOUNT": "Loan amount in $USD",
        "INCOME": "Household income in $USD",
        "INCOME_LOAN_RATIO": "Ratio of LOAN_AMOUNT/INCOME",
        "MEAN_COUNTY_INCOME": "Average household income aggregated at county level",
        "HIGH_INCOME_FLAG": "Binary flag to indicate whether household income is higher than MEAN_COUNTY_INCOME",
        "AVG_THIRTY_DAY_LOAN_AMOUNT": "Rolling 30 day average of LOAN_AMOUNT"
    }
)

loan_fv = fs.register_feature_view(loan_fv, version=VERSION_NUM, overwrite=True)

In [ ]:
fs.list_feature_views()

In [ ]:
#Create link to feature store UI to inspect newly created feature view!
org_name = session.sql('SELECT CURRENT_ORGANIZATION_NAME()').collect()[0][0]
account_name = session.sql('SELECT CURRENT_ACCOUNT_NAME()').collect()[0][0]

print(f'https://app.snowflake.com/{org_name}/{account_name}/#/features/database/{DB}/store/{SCHEMA}')

## Retrieve a dataset and teh feature view

In [ ]:
ds = fs.generate_dataset(
    name=f"MORTGAGE_DATASET_EXTENDED_FEATURES_{VERSION_NUM}",
    spine_df=df.select("LOAN_ID", "TIMESTAMP", "LOAN_PURPOSE_NAME","MORTGAGERESPONSE"), #only need the features used to fetch rest of feature view
    features=[loan_fv],
    spine_timestamp_col="TIMESTAMP",
    spine_label_cols=["MORTGAGERESPONSE"]
)


In [ ]:
ds_sp = ds.read.to_snowpark_dataframe()
ds_sp.show(5)

In [ ]:
feature_view = fs.get_feature_view("Mortgage_Feature_View", version=VERSION_NUM)

In [ ]:
# -----------------------------------------------
# 3. GENERATE TRAINING SET
# -----------------------------------------------

label_col = "MORTGAGERESPONSE" 

training_set = fs.generate_training_set(
    spine_df=feature_df,
    features=[feature_view],
    spine_label_cols=label_col,
)


In [ ]:
type(training_set)
#train_pd = DataConnector.from_dataframe(
#    training_set.read.to_snowpark_dataframe()
#).to_pandas()

In [ ]:
train_pd = training_set.to_pandas()